In [ ]:
# Libraries for use in different cells
import sys 
from shapely import Point, wkt, LineString
from shapely.geometry import shape
from shapely import wkb
from shapely.ops import substring
import pandas as pd
import overturemaps as om
import geopandas as gpd
import folium
import numpy as np
import ast
import re
import matplotlib.pyplot as plt
import requests
import warnings

In [ ]:
# Parameters for selection of data
rootPath = 'C:\\Data\\GitHub\\jetgeo\\OM2ANY'
areaName = 'Hamar'
areaOfInterestFile = 'aoi.geojson'

In [ ]:
# Read GeoJSON Polygon, transform to EPSG:5973 and use for filtering NVDB
gjFile = rootPath + '\\Data\\' + areaName + '\\' + areaOfInterestFile

# Bounding box
def get_bbox(geometry):
    polygon = shape(geometry)
    return polygon.bounds

gjP4326 = gpd.read_file(gjFile)
gjP5973 = gjP4326.to_crs(epsg=5973)

# Create a string with the coordinates separated by commas
polygon_coords = gjP5973.geometry.iloc[0].exterior.coords
coords_string = ", ".join([f"{x:.6f} {y:.6f}" for x, y in polygon_coords])
print(f"Coordinates of the polygon feature: {coords_string}")

# Create a  bounding box string in EPSG:4326
geometry = gjP4326.geometry[0]
minlon, minlat, maxlon, maxlat = geometry.bounds
bbox_string = minlon, minlat, maxlon, maxlat
print('Bounding box string WGS84 lat lon:' + str(bbox_string))

#Show the area on a map
gjP4326.explore() #(width=800, height=600)

In [ ]:
def rbr2GDF(rbr):
    # Convert Record Batch Reader to GeoDataFrame
    # Create a Pandas DataFrame with the geometries 
    df = rbr.to_pandas()
    # Ensure the geometry column is correctly interpreted as shapely objects
    df['geometry'] = df['geometry'].apply(wkb.loads)

    # Convert any ndarray columns to lists
    for col in df.columns:
        if isinstance(df[col].iloc[0], np.ndarray):
            # df[col] = df[col].apply(lambda x: x.tolist())
            df[col] = df[col].apply(lambda x: x.tolist() if x is not None else x)

    # Convert any other non-serializable types to strings
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: str(x) if isinstance(x, (np.ndarray, list, dict)) else x)

    # DataFrame to GeoDataFrame, set CRS
    gdf = gpd.GeoDataFrame(df,geometry='geometry',crs="EPSG:4326")
    return gdf


#---------------------------------------------------------------------
print('Reading segments')
ft= "segment"
segTable = om.record_batch_reader(ft, bbox_string).read_all()
segTable = segTable.combine_chunks()
segGDF = rbr2GDF(segTable)
print('Read ' + str(len(segGDF)) + ' segments within the bounding box')

# Keep only MODI-relevant segments
filtered_segGDF = segGDF[
    # (segGDF['class'].isin(['motorway', 'trunk', 'primary', 'secondary', 'tertiary']))]
    (segGDF['class'].isin(['motorway', 'trunk', 'primary', 'secondary', 'tertiary','residential']))]
segGDF = filtered_segGDF
print('Filtered to ' + str(len(segGDF)) + ' segments of relevant type')

# Intersect with AOI-polygon
segGDF_clipped = gpd.sjoin(segGDF, gjP4326, predicate='intersects')
segGDF = segGDF_clipped
print('Filtered to ' + str(len(segGDF_clipped)) + ' segments within the polygon')

# Get the bounding box of the segments dataset, for use in selection of connectors
bounds = segGDF.total_bounds  # [minx, miny, maxx, maxy]
bbox_string = bounds[0], bounds[1], bounds[2], bounds[3]

print('Reading connectors')
ft= "connector"
conTable = om.record_batch_reader(ft, bbox_string).read_all()
conGDF = rbr2GDF(conTable)
print('Read ' + str(len(conGDF)) + ' connectors within the bounding box')
#Remove connectors that are not related to any segment
# Start with extracting a unique list of connectors related to a segment
unique_connector_ids = set()
# Iterate through each row in the 'connectors' column of segGDF
for _, seg_row in segGDF.iterrows():
    # Convert the string representation of the list to an actual list of dictionaries
    connectors_str = seg_row['connectors']
    connectors_list = ast.literal_eval(connectors_str)
    # Iterate through each dictionary in the list and add the connector_id to the set
    for connector in connectors_list:
        unique_connector_ids.add(connector['connector_id'])
# Convert the set to a list to get the final list of unique connector_id values
unique_connector_ids = list(unique_connector_ids)
filtered_conGDF = conGDF[conGDF['id'].isin(unique_connector_ids)]
conGDF=filtered_conGDF
print('Filtered to ' + str(len(filtered_conGDF)) + ' connectors that are connected to any of the filtered segments')

#Export to GeoJSON, for use in QGIS
gdf_filtered = segGDF[['id', 'geometry']]
gdf_filtered.to_file(rootPath + "\\data\\" + areaName + "\\om_segments.geojson", driver="GeoJSON")
gdf_filtered = conGDF[['id', 'geometry']]
gdf_filtered.to_file(rootPath + "\\data\\" + areaName + "\\om_connectors.geojson", driver="GeoJSON")
print('Exported to GeoJSON')


In [ ]:
#Show the OM data on the map
m=segGDF.explore(tiles='CartoDB positron',column = 'class', cmap='tab10', name="OM segments", width=1400)
conGDF.explore(m=m, color='Grey', name="OM nodes")
folium.LayerControl().add_to(m)
# Display the map
m